# V2X and Cooperative Driving

**Module 07 · Runtime Assurance and Operations** · Notebook 2 of 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/07_Runtime_Assurance_and_Operations/02_v2x_cooperative_driving.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~50 min | [Sensor fusion](../01_Perception/06_sensor_fusion.ipynb), [ISO/SAE 21434 fundamentals](../05_Cybersecurity_ISO21434/01_iso21434_fundamentals.ipynb) | SAE J2735, ETSI EN 302 637-2/-3, ETSI TS 103 324, IEEE 1609.2, ETSI TS 103 097, 3GPP TS 22.185/22.186 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

After this notebook you will be able to:

- distinguish V2V, V2I, V2P and V2N, and the radio technologies ITS-G5/DSRC, LTE-V2X and NR-V2X (PC5 sidelink vs. Uu);
- name the main message sets in the US (SAE J2735: BSM, SPaT, MAP, PSM, SDSM) and in Europe (ETSI CAM, DENM, SPATEM/MAPEM, CPM);
- quote the 3GPP latency and reliability requirements correctly, and explain why "V2X range" is not a single number;
- simulate packet delivery versus distance and channel load and its effect on a cooperative braking warning;
- describe V2X security (IEEE 1609.2 / ETSI TS 103 097, SCMS / EU CCMS, pseudonym certificates, misbehaviour detection);
- argue where V2X belongs in a safety case: as a non-safety-critical *additional* input, or as an input that then carries safety requirements.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
      "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "black": "#000000"}
pd.set_option("display.max_colwidth", 90)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
## 1. What V2X adds

**V2X (vehicle-to-everything)** covers communication with other vehicles (V2V), infrastructure such as traffic lights (V2I), vulnerable road users' devices (V2P) and the network/cloud (V2N). Its distinctive contribution is information that on-board sensors cannot provide:

- **non-line-of-sight awareness** — a vehicle braking hard behind a truck or around a corner;
- **intent and state that are not observable** — signal phase and timing, an emergency vehicle's route, a braking event before the brake lights are visible;
- **perception beyond sensor range**, e.g. hazard warnings from vehicles kilometres ahead.

It does not replace on-board perception: messages can be lost, late, wrong or malicious, and only equipped road users are visible. The driverless services operating today (see [notebook 03](03_industry_deployment.ipynb)) rely on on-board sensing and maps, not on V2X.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
for ax, title, known in [(axes[0], "Without V2X: the braking car is occluded", False),
                         (axes[1], "With V2X: DENM/CAM from the braking car", True)]:
    ax.set_xlim(0, 100); ax.set_ylim(0, 20); ax.axis("off"); ax.set_title(title)
    ax.add_patch(plt.Rectangle((8, 8), 8, 4, color=OI["blue"])); ax.text(12, 10, "ego", ha="center", va="center", color="white")
    ax.add_patch(plt.Rectangle((35, 7), 18, 6, color="grey")); ax.text(44, 10, "truck", ha="center", va="center", color="white")
    ax.add_patch(plt.Rectangle((70, 8), 8, 4, color=OI["red"], alpha=1.0 if known else 0.3))
    ax.text(74, 5.5, "braking car\n(" + ("known via V2X" if known else "hidden") + ")", ha="center", fontsize=9)
    ax.plot([16, 35], [10, 10], ls="--", color=OI["blue"]); ax.text(25, 12, "line of sight blocked", ha="center", fontsize=8)
    if known:
        ax.annotate("", xy=(16, 16), xytext=(70, 16), arrowprops=dict(arrowstyle="->", color=OI["green"], lw=2))
        ax.text(43, 17.3, "radio link (non-line-of-sight)", ha="center", fontsize=8, color=OI["green"])
plt.tight_layout(); plt.show()

---
## 2. Radio technologies and regulatory status

Two families operate in the 5.9 GHz ITS band. Both support **direct** communication without a network; cellular V2X additionally offers network-based **V2N** via the Uu interface.

In [ ]:
access = pd.DataFrame([
    ["Radio basis", "IEEE 802.11p (OFDM, CSMA/CA); successor IEEE 802.11bd",
     "3GPP Rel-14/15 LTE sidelink (PC5), SC-FDMA, sensing-based semi-persistent scheduling",
     "3GPP Rel-16+ NR sidelink (PC5); complements, does not replace, LTE-V2X basic safety"],
    ["Spectrum", "5.9 GHz ITS band", "5.9 GHz ITS band (PC5); operator bands for Uu", "5.9 GHz ITS band (PC5); operator bands for Uu"],
    ["Network needed?", "No (ad hoc broadcast)", "No for PC5 'mode 4'; Uu uses the cellular network (V2N)",
     "No for PC5; Uu via network"],
    ["Main use", "Day-1 awareness and hazard warnings (CAM/DENM, BSM)", "Same message sets as left (BSM/CAM/DENM)",
     "Advanced use cases: sensor sharing, manoeuvre coordination, platooning"],
    ["Standards", "IEEE 802.11p; ETSI EN 302 663 (ITS-G5); US DSRC (being retired)",
     "3GPP TS 36.300 ff.; ETSI EN 303 613; SAE J3161/1", "3GPP TS 38.300 ff."],
], columns=["", "ITS-G5 / DSRC", "LTE-V2X (C-V2X)", "NR-V2X (5G sidelink)"])
access

**Status (as of October 2026).**

- **United States.** In 2020 the FCC kept only the upper 30 MHz (5.895–5.925 GHz) for ITS and opened the lower 45 MHz (5.850–5.895 GHz) to unlicensed use. Its *Second Report and Order* (FCC 24-123, adopted 20 November 2024) designates C-V2X as the ITS technology in the upper band and requires DSRC operations to transition to C-V2X or cease **two years after publication in the Federal Register** (published 13 December 2024, i.e. mid-December 2026). Source: [FCC 24-123](https://docs.fcc.gov/public/attachments/FCC-24-123A1_Rcd.pdf); Federal Register document 2024-28980. We found no extension of this deadline as of October 2026; check FCC ET Docket 19-138 before teaching the exact date. In August 2024 USDOT published *Saving Lives with Connectivity: A Plan to Accelerate V2X Deployment*.
- **Europe.** ETSI standardises both an ITS-G5 access layer (EN 302 663) and an LTE-V2X access layer (EN 303 613); the message sets and security above the access layer are shared. EU policy for C-ITS is technology-neutral at the access layer.
- **Everywhere:** the message sets and the security framework matter more for interoperability than the radio.

### 2.1 Latency and reliability — what the requirements actually say

Latency requirements are set per use case, not per technology. The 3GPP service requirements are:

In [ ]:
req = pd.DataFrame([
    ["TS 22.185 (LTE V2X, Rel-14)", "V2V / V2P messages, direct or via RSU", 100, None, "R-5.2.1-001"],
    ["TS 22.185 (LTE V2X, Rel-14)", "Pre-crash sensing (V2V)", 20, None, "R-5.2.1-002"],
    ["TS 22.185 (LTE V2X, Rel-14)", "V2N (via application server)", 1000, None, "R-5.2.1-004"],
    ["TS 22.186 (eV2X, Rel-16)", "Cooperative collision avoidance", 10, 99.99, "R.5.3-001"],
    ["TS 22.186 (eV2X, Rel-16)", "Emergency trajectory alignment", 3, 99.999, "R.5.3-006"],
    ["TS 22.186 (eV2X, Rel-16)", "Sensor sharing, higher automation, 200 m", 3, 99.999, "R.5.4-003"],
    ["TS 22.186 (eV2X, Rel-16)", "Sensor sharing, lower automation, 1000 m", 100, 99.0, "R.5.4-001"],
], columns=["3GPP spec", "Scenario", "max end-to-end latency [ms]", "reliability [%]", "requirement"])
req

Notes:

- The **100 ms** requirement for basic V2V safety messages (TS 22.185) applies to *both* families; IEEE 802.11p channel access is typically far faster than this when the channel is not congested. A claim such as "DSRC 100 ms vs. 5G 20 ms" compares a requirement with a marketing figure and is not meaningful.
- Single-digit-millisecond, 99.99–99.999 % requirements (TS 22.186) apply to advanced eV2X use cases such as emergency trajectory alignment, for which NR-V2X was designed.
- Payload sizes of awareness messages are a few hundred bytes; **channel congestion**, not raw bandwidth, is the limiting factor. Decentralised congestion control (ETSI TS 102 687; SAE J2945/1 in the US) adapts message rate and power.

---
## 3. Message sets

The US message dictionary is **SAE J2735** with performance requirements in the **SAE J2945/x** series; Europe uses the **ETSI** facilities-layer standards. They are functionally similar but not identical.

In [ ]:
msgs = pd.DataFrame([
    ["US", "BSM - Basic Safety Message", "SAE J2735 (format), J2945/1 (performance)", "periodic (nominally 10 Hz)", "position, speed, heading, acceleration, brake status, size"],
    ["US", "SPaT / MAP", "SAE J2735", "periodic (RSU)", "signal phase & timing; intersection geometry"],
    ["US", "PSM - Personal Safety Message", "SAE J2735", "periodic", "vulnerable road user position/motion"],
    ["US", "SDSM - Sensor Data Sharing Message", "SAE J3224 (format in J2735)", "periodic", "objects detected by the sender's sensors"],
    ["EU", "CAM - Cooperative Awareness Message", "ETSI EN 302 637-2", "every 100-1000 ms, triggered by dynamics", "as BSM"],
    ["EU", "DENM - Decentralized Environmental Notification", "ETSI EN 302 637-3", "event-driven, repeated", "hazard type, location, validity (e.g. emergency braking, road works)"],
    ["EU", "SPATEM / MAPEM", "ETSI TS 103 301", "periodic (RSU)", "signal phase & timing; intersection topology"],
    ["EU", "CPM - Collective Perception Message", "ETSI TS 103 324 (V2.1.1, 2023)", "every 100-1000 ms", "perceived objects, sensor information, free space"],
], columns=["Region", "Message", "Standard", "Generation", "Content"])
msgs

A CAM is not sent at a fixed rate. Per ETSI EN 302 637-2 (V1.4.1), a CAM is generated when, since the last CAM, the heading has changed by more than 4°, the position by more than 4 m or the speed by more than 0.5 m/s — but never more often than every 100 ms (T_GenCamMin) and at least every 1000 ms (T_GenCamMax). The simplified implementation below (checking every 50 ms) shows how the rate follows the vehicle dynamics:

In [ ]:
# ETSI EN 302 637-2 (V1.4.1): a CAM is generated when, since the last CAM, heading changed > 4 deg,
# position changed > 4 m or speed changed > 0.5 m/s - but not more often than every 100 ms (T_GenCamMin)
# and at least every 1000 ms (T_GenCamMax). Simplified check, evaluated every 100 ms.
def cam_rate(speed_mps, heading_rate_dps=0.0, accel=0.0, horizon=60.0):
    dt = 0.05   # T_CheckCamGen <= T_GenCamMin (100 ms)
    last_pos = pos = 0.0
    last_head = head = 0.0
    last_v = v = speed_mps
    last_t, n = 0.0, 0
    for k in range(1, int(horizon / dt) + 1):
        tt = k * dt
        pos += v * dt; head += heading_rate_dps * dt; v = max(0.0, v + accel * dt)
        trig = abs(head - last_head) > 4 or abs(pos - last_pos) > 4 or abs(v - last_v) > 0.5
        if (trig and tt - last_t >= 0.1 - 1e-9) or tt - last_t >= 1.0 - 1e-9:
            n += 1; last_t, last_pos, last_head, last_v = tt, pos, head, v
    return n / horizon

cam = pd.DataFrame([{"situation": s, "speed [km/h]": v, "CAM rate [Hz]": round(cam_rate(v / 3.6, hr, a), 1)}
                    for s, v, hr, a in [("standstill", 0, 0, 0), ("slow urban", 10, 0, 0), ("urban", 30, 0, 0),
                                        ("urban, turning 20 deg/s", 30, 20, 0), ("motorway", 130, 0, 0)]])
cam

**Cooperative perception.** With the CPM (ETSI TS 103 324) or SDSM (SAE J3224), vehicles and roadside units share *objects they have detected*, not only their own state. The receiver must transform the objects into its own frame, compensate the message age using the timestamps (millisecond-level alignment, with state extrapolation) and fuse them like any other sensor track — including the sender's stated uncertainty. CPMs, like CAMs, are signed (TS 103 324 requires signing with authorisation tickets), but a signature authenticates the *sender*, not the *correctness* of the perceived objects.

---
## 4. Simulation: packet delivery and a cooperative braking warning

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements. The link model (log-distance path loss, log-normal shadowing, load-dependent collisions) is deliberately simple and not calibrated to any product.

### 4.1 Packet delivery ratio (PDR) vs. distance and channel busy ratio (CBR)

In [ ]:
# Illustrative link model (NOT calibrated to a specific radio): log-distance path loss with log-normal
# shadowing, a fixed SINR threshold, and a collision probability that grows with the channel busy ratio (CBR).
P_TX_DBM, NOISE_DBM, SINR_TH_DB = 29.0, -98.0, 5.0   # EIRP incl. antenna gains, noise floor (10 MHz, NF 6 dB), threshold
PL0_DB, D0_M = 47.9, 1.0                             # free-space loss at 1 m, 5.9 GHz
EXPONENT = {"LOS (motorway)": 2.4, "NLOS (blocked by a truck)": 3.0}
SIGMA_DB = {"LOS (motorway)": 4.0, "NLOS (blocked by a truck)": 6.0}

def link_ok(d_m, env, n, rng=rng):
    """True where the received SNR (one shadowing draw per sample) exceeds the decoding threshold."""
    pl = PL0_DB + 10 * EXPONENT[env] * np.log10(np.maximum(d_m, D0_M) / D0_M) + rng.normal(0, SIGMA_DB[env], n)
    return (P_TX_DBM - pl - NOISE_DBM) > SINR_TH_DB

def p_collision(d_m, cbr):
    """Packet loss from simultaneous transmissions; grows with load and with distance (hidden terminals)."""
    return cbr ** 2 * (0.2 + d_m / 1000)

def pdr(d_m, cbr, env, n=4000, rng=rng):
    """Monte Carlo packet delivery ratio at distance d_m for channel busy ratio cbr in [0,1]."""
    ok = link_ok(d_m, env, n, rng) & (rng.random(n) > p_collision(d_m, cbr))
    return ok.mean()

distances = np.arange(25, 1001, 25)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
pdr_tab = {}
for ax, env in zip(axes, EXPONENT):
    for cbr, c in [(0.1, OI["green"]), (0.4, OI["orange"]), (0.7, OI["red"])]:
        y = [pdr(d, cbr, env) for d in distances]
        pdr_tab[(env, cbr)] = dict(zip(distances, y))
        ax.plot(distances, y, color=c, lw=2, label=f"CBR = {cbr:.0%}")
    ax.set_title(env); ax.set_xlabel("distance [m]"); ax.axhline(0.9, color="grey", ls=":", lw=1)
axes[0].set_ylabel("packet delivery ratio [-]"); axes[0].legend()
fig.suptitle("Illustrative PDR vs distance and channel load (synthetic link model)")
plt.tight_layout(); plt.show()

In [ ]:
def range_at(env, cbr, target=0.9):
    d = [dd for dd, p in pdr_tab[(env, cbr)].items() if p >= target]
    return max(d) if d else 0

summary = pd.DataFrame([{"environment": env, "CBR": cbr, "max distance with PDR >= 90 % [m]": range_at(env, cbr)}
                        for env in EXPONENT for cbr in (0.1, 0.4, 0.7)])
display(summary)
for env in EXPONENT:
    lo, hi = range_at(env, 0.1), range_at(env, 0.7)
    if hi == 0:
        print(f"{env}: 90 % PDR is reached up to {lo} m at CBR 10 %, but at no distance at CBR 70 %.")
    else:
        print(f"{env}: the 90 %-PDR range shrinks from {lo} m at CBR 10 % to {hi} m at CBR 70 %.")
print("Hence 'the range of V2X' is not a single number: it depends on propagation and on channel load,")
print("which is why congestion control (ETSI TS 102 687, SAE J2945/1) adapts message rate and power.")

### 4.2 Effect on a cooperative braking warning

A hidden car brakes hard at 8 m/s²; the truck behind it reacts after 1.5 s; the ego vehicle follows the truck at 15 m (≈0.5 s at 100 km/h). Without V2X, ego reacts 0.6 s after the truck starts braking. With V2X, the braking car broadcasts a warning (e.g. a DENM for emergency braking) every 100 ms; ego reacts 0.3 s after the first successfully received packet, or to the truck, whichever comes first.

Two loss models are compared: **independent** losses per packet, and **correlated** losses, where shadowing is drawn once per event (the truck keeps blocking the path for the whole manoeuvre) and only collisions vary per packet.

In [ ]:
# Cooperative braking warning: car C (hidden) brakes hard; truck T follows C; ego E follows T.
# Without V2X, ego reacts to the truck's brake lights / deceleration. With V2X, ego may react to C's DENM.
# Illustrative parameters (synthetic, chosen for teaching).
V0 = 100 / 3.6         # m/s, all vehicles
A_C, A_T, A_E = 8.0, 6.0, 6.0    # braking decelerations of car, truck, ego [m/s^2]
TRUCK_REACT = 1.5      # s, truck driver reaction to the car braking
EGO_SENSOR_REACT = 0.6 # s, ego perception + actuation after the truck starts braking
EGO_V2X_REACT = 0.3    # s, ego processing + actuation after a V2X warning is received
GAP_TE = 15.0          # m, gap truck -> ego (~0.5 s time gap: close following)
MSG_PERIOD = 0.1       # s, DENM repetition interval
LATENCY = 0.05         # s, end-to-end latency per packet

def min_gap(ego_brake_start):
    """Minimum truck-ego gap [m] when ego starts braking at ego_brake_start (s after car braking)."""
    tt = np.arange(0, 15, 0.01)
    def pos(t0, a):  # position relative to start of braking (constant speed before t0)
        tb = np.clip(tt - t0, 0, V0 / a)
        return V0 * np.minimum(tt, t0) + V0 * tb - 0.5 * a * tb ** 2 + 0 * tt
    x_t = pos(TRUCK_REACT, A_T)
    x_e = pos(ego_brake_start, A_E)
    return np.min(GAP_TE + x_t - x_e)

def first_reception(env, cbr, d_ce, rng, correlated=True, n_rep=30):
    """Time of first successful reception of the repeated warning (inf if none within 3 s).

    correlated=True: one shadowing draw for the whole event (the truck keeps blocking the path),
    collisions independent per packet. correlated=False: every packet sees independent shadowing.
    """
    snr_ok = np.repeat(link_ok(d_ce, env, 1, rng), n_rep) if correlated else link_ok(d_ce, env, n_rep, rng)
    ok = snr_ok & (rng.random(n_rep) > p_collision(d_ce, cbr))
    k = np.flatnonzero(ok)
    return (k[0] * MSG_PERIOD + LATENCY) if k.size else np.inf

t_sensor = TRUCK_REACT + EGO_SENSOR_REACT
gap_sensor_only = min_gap(t_sensor)
results = []
# The radio distance d_ce between the braking vehicle and ego is varied: the hazard may be one or several
# vehicles ahead (e.g. the end of a queue); the kinematics of the truck-ego pair are kept fixed.
for env in EXPONENT:
    for d_ce in (60, 150, 250):
        for cbr in (0.1, 0.7):
            p = pdr(d_ce, cbr, env)
            gaps, gains = [], []
            for _ in range(2000):
                t_rx = first_reception(env, cbr, d_ce, rng)
                t_brake = min(t_sensor, t_rx + EGO_V2X_REACT)   # ego uses whichever cue comes first
                gaps.append(min_gap(t_brake)); gains.append(t_sensor - t_brake)
            gaps, gains = np.array(gaps), np.array(gains)
            iid_none = np.mean([np.isinf(first_reception(env, cbr, d_ce, rng, correlated=False)) for _ in range(2000)])
            results.append({"environment": env, "distance [m]": d_ce, "CBR": cbr, "PDR": round(p, 3),
                            "P(all 30 lost), independent": round(iid_none, 4),
                            "P(no V2X benefit), correlated": round(np.mean(gains <= 0), 4),
                            "P(collision)": round(np.mean(gaps <= 0), 4),
                            "median reaction gain [s]": round(np.median(gains), 2)})
warn = pd.DataFrame(results)
print(f"Sensor-only reference: ego brakes {t_sensor:.1f} s after the car; minimum gap {gap_sensor_only:.1f} m "
      f"({'collision' if gap_sensor_only <= 0 else 'no collision'}).")
warn

**Text summary** (derived from the results):

In [ ]:
print("Text summary (derived from the table above):")
best, worst = warn.iloc[warn["P(collision)"].idxmin()], warn.iloc[warn["P(collision)"].idxmax()]
print(f"- Sensor-only: minimum gap {gap_sensor_only:.1f} m -> {'collision' if gap_sensor_only <= 0 else 'no collision'}.")
print(f"- With V2X, P(collision) ranges from {best['P(collision)']:.2%} ({best['environment']}, {best['distance [m]']} m, CBR {best['CBR']:.0%}) "
      f"to {worst['P(collision)']:.2%} ({worst['environment']}, {worst['distance [m]']} m, CBR {worst['CBR']:.0%}).")
print(f"- Median reaction-time gain from V2X (when received): {warn['median reaction gain [s]'].max():.2f} s.")
w = warn.iloc[warn["P(no V2X benefit), correlated"].idxmax()]
print(f"- Repetition defeats independent losses (P(all lost) <= {warn['P(all 30 lost), independent'].max():.4f}), but with "
      f"correlated shadowing the warning is missed in up to {w['P(no V2X benefit), correlated']:.1%} of events "
      f"({w['environment']}, {w['distance [m]']} m, CBR {w['CBR']:.0%}). Loss correlation, not average PDR, limits the benefit.")
if gap_sensor_only <= 0:
    print("- The sensor-only system collides in this scenario, so a safety argument that relied on V2X would make")
    print("  PDR and latency safety requirements; whenever all packets are lost the outcome falls back to the sensor-only case.")
else:
    print("- The sensor-only system already avoids the collision: here V2X adds margin and comfort, not the safety baseline.")

**What the simulation shows.** Repetition makes the warning robust against *independent* losses, so average PDR is a misleading metric. What limits the benefit is *correlated* loss — blockage by a large vehicle, being out of range, or a jammer — which removes the whole warning at once. Exactly in those events the outcome reverts to the sensor-only case. That is the key input for the safety perspective below.

---
## 5. V2X security

V2X security is specified by **IEEE 1609.2** (US) and **ETSI TS 103 097** (EU; it profiles IEEE 1609.2 data structures), with trust and privacy management in **ETSI TS 102 941**. Every safety message is digitally signed (ECDSA) with a short-lived certificate chained to a trusted root:

- **US: SCMS** (Security Credential Management System);
- **EU: EU CCMS** (C-ITS Credential Management System) under a common European certificate policy and trust list.

**Privacy.** Vehicles sign with **pseudonym certificates** (called *authorisation tickets* in ETSI). TS 102 941 sets the aims of *pseudonymity* and *unlinkability*: an ITS station changes its pseudonym — together with its station ID and other identifiers — so that messages cannot be linked into a long-term track, while the PKI can still resolve misbehaviour. Concrete change intervals are set by the applicable profiles and certificate policies.

**Engineering process.** These are *security controls*; the process for identifying threats and deciding on them is ISO/SAE 21434 (TARA), see [Module 05](../05_Cybersecurity_ISO21434/02_tara.ipynb). CALs are assigned per cybersecurity goal (informative Annex E), not per threat, so they are not shown here.

| Threat | Example | Primary controls |
|---|---|---|
| Spoofing by an outsider | forged emergency-vehicle message | signature + certificate-chain verification (1609.2 / TS 103 097) |
| Insider / compromised sender | validly signed but false data | plausibility checks, misbehaviour detection and reporting, certificate revocation |
| Replay | re-broadcast of an old DENM | generation time and location checks in the security header; DENM validity |
| Sybil | one attacker using many pseudonyms | issuance limits on simultaneously valid pseudonyms; misbehaviour detection |
| Jamming | noise on the 5.9 GHz channel | detection and graceful fallback to on-board sensing (802.11p and PC5 do not use frequency hopping) |
| Tracking (privacy) | linking messages of one vehicle | pseudonym and identifier changes |

The code below shows the simplest layer of misbehaviour detection: kinematic plausibility checks on a received track. These checks matter precisely because a compromised sender produces messages with valid signatures.

In [ ]:
# Misbehaviour detection: kinematic plausibility checks on received CAM/BSM tracks.
# A validly signed message from a compromised (insider) sender passes signature verification,
# so receivers additionally check physical plausibility. Synthetic tracks, 10 Hz.
def track(rng, n=50, v=20.0, spoof=None):
    x = np.cumsum(np.full(n, v * 0.1)) + rng.normal(0, 0.3, n)      # GNSS noise ~0.3 m
    speed = np.full(n, v) + rng.normal(0, 0.2, n)
    if spoof == "teleport":
        x[25:] += 60.0                                             # sudden position jump
    elif spoof == "ghost_brake":
        speed[25:] = np.maximum(0, v - 15 * np.arange(1, n - 24) * 0.1)  # claims 15 m/s^2 braking
        x[25:] = x[24] + np.cumsum(speed[25:] * 0.1)
    elif spoof == "inconsistent":
        speed[25:] += 12.0                                         # speed field disagrees with positions
    return x, speed

def plausibility(x, speed, dt=0.1, a_max=12.0, tol=3.0):
    v_from_pos = np.diff(x) / dt
    accel = np.diff(speed) / dt
    checks = {"position jump": np.any(np.abs(np.diff(x)) > (speed[:-1] * dt + 0.5 * a_max * dt ** 2 + 2.0)),
              "acceleration > 12 m/s^2": np.any(np.abs(accel) > a_max),
              "speed vs. position mismatch": np.any(np.abs(np.convolve(v_from_pos - speed[1:], np.ones(5) / 5, "valid")) > tol)}
    return checks

rows = []
for case in [None, "teleport", "ghost_brake", "inconsistent"]:
    x, s = track(rng, spoof=case)
    chk = plausibility(x, s)
    rows.append({"track": case or "genuine", **chk, "flagged": any(chk.values())})
plaus = pd.DataFrame(rows)
display(plaus)
fp = plaus.loc[plaus["track"] == "genuine", "flagged"].item()
caught = plaus.loc[plaus["track"] != "genuine", "flagged"].sum()
print(f"{caught} of {len(plaus) - 1} manipulated tracks flagged; genuine track flagged: {fp}.")
print("A spoofer who keeps the claimed track physically consistent would pass these checks; cross-checking against")
print("on-board sensors and misbehaviour reporting to the PKI (certificate revocation) are the next layers.")

---
## 6. Safety perspective: where does V2X sit in the safety argument?

There are two coherent positions — and one incoherent one.

1. **V2X as a non-safety-critical additional input.** The safety case is complete *without* V2X: the system meets its safety goals and SOTIF acceptance criteria with on-board sensing alone. V2X can improve comfort, efficiency and margins (earlier, gentler braking), but its loss never makes the system less safe than the baseline. Wrong V2X data must not trigger hazardous actions on its own: high-authority actuation (e.g. emergency braking) requires on-board confirmation, while low-authority reactions (pre-filling brakes, a warning, a slight speed reduction) may be taken on V2X alone.
2. **V2X as a credited safety input.** If the argument relies on V2X (e.g. to cover occlusions that sensors cannot), then V2X availability, latency, integrity and authenticity become **safety requirements**: correlated-loss scenarios, the infrastructure's own integrity, the PKI and the equipment rate of other road users all enter the safety case. Lost or wrong messages are then triggering conditions in the sense of ISO 21448; malfunctions in the V2X chain fall under ISO 26262; attacks under ISO/SAE 21434.
3. **Incoherent:** claiming a safety benefit from V2X while treating its failures as out of scope.

The simulation above makes the trade-off concrete: in correlated-loss events, the system that *credits* V2X falls back to the sensor-only outcome — which, in our scenario, is a collision. A design that relies on V2X therefore needs either guaranteed coverage (e.g. infrastructure-based sensing at a specific intersection, within a defined ODD) or a behavioural fallback, such as a larger following distance when no V2X coverage is confirmed. The ODD can state V2X as a requirement — ISO 34503 lists *connectivity* as an environmental condition — which brings it under the runtime monitoring of [notebook 01](01_odd_and_runtime_monitoring.ipynb).

---
## Key takeaways

1. V2X provides non-line-of-sight information and intent (e.g. SPaT); it complements, and does not replace, on-board perception. Today's driverless deployments do not depend on it.
2. ITS-G5/DSRC and LTE-V2X both serve basic safety messages under a 100 ms latency requirement; NR-V2X targets advanced use cases (3–10 ms, up to 99.999 %). In the US, DSRC must cease about two years after the FCC's Second Report and Order was published (mid-December 2026).
3. Message sets: SAE J2735 (BSM, SPaT, MAP, PSM, SDSM/J3224) and ETSI (CAM, DENM, SPATEM/MAPEM, CPM). CAM rates follow vehicle dynamics (100–1000 ms).
4. Range and reliability depend on propagation and channel load; correlated losses, not average PDR, limit the safety benefit.
5. Security: IEEE 1609.2 / ETSI TS 103 097 signatures, SCMS / EU CCMS trust, pseudonym certificates for privacy, and plausibility-based misbehaviour detection against insiders; the engineering process is ISO/SAE 21434.
6. Decide explicitly whether V2X is a non-safety-critical add-on or a credited safety input — the second turns communication performance into safety requirements.

## Self-assessment

1. What is the difference between PC5 and Uu in C-V2X?
2. Which 3GPP requirement sets the 100 ms latency, and to which use cases does a 3 ms requirement apply?
3. Name the European equivalents of BSM and of SAE J3224 SDSM, and the event-driven ETSI message for hazards.
4. In the braking simulation, why does the probability that V2X brings no benefit differ so much between the independent and the correlated loss models?
5. Why is a valid signature not sufficient to trust the content of a V2X message?
6. Give one argument for treating V2X as a non-safety-critical input, and state what changes if it is credited in the safety case.

<details>
<summary>Answers</summary>

1. PC5 is the direct sidelink between devices (no network needed); Uu is the link to the cellular base station, used for V2N.
2. 3GPP TS 22.185 (R-5.2.1-001: V2V/V2P messages, 100 ms). TS 22.186 specifies 3 ms (with 99.999 % reliability) for e.g. emergency trajectory alignment and sensor sharing for higher degrees of automation.
3. CAM (≈ BSM) and CPM (≈ SDSM); DENM for event-driven hazard warnings.
4. With independent losses, 30 repetitions almost never all fail. With correlated losses (persistent blockage), one bad link state removes all repetitions at once, so the probability of no benefit approaches the probability of the bad link state.
5. The signature proves that the sender holds a valid certificate; a compromised or faulty sender can still sign false data. Plausibility checks, cross-checks with on-board sensors and misbehaviour reporting are needed.
6. Argument: equipment rates and coverage are incomplete and losses are correlated, so a safety case that does not depend on V2X is more robust. If V2X is credited, its availability, latency, integrity and authenticity become safety requirements and must be verified and monitored (e.g. as an ODD condition).
</details>

## References

- 3GPP TS 22.185 V14.3.0 (2017). *Service requirements for V2X services*. https://www.etsi.org/deliver/etsi_ts/122100_122199/122185/14.03.00_60/ts_122185v140300p.pdf
- 3GPP TS 22.186 V16.2.0 (2020). *Enhancement of 3GPP support for V2X scenarios*. https://www.etsi.org/deliver/etsi_ts/122100_122199/122186/16.02.00_60/ts_122186v160200p.pdf
- ETSI EN 302 637-2 V1.4.1 (2019). *Cooperative Awareness Basic Service*. https://www.etsi.org/deliver/etsi_en/302600_302699/30263702/01.04.01_60/en_30263702v010401p.pdf
- ETSI EN 302 637-3 V1.3.1 (2019). *Decentralized Environmental Notification Basic Service*.
- ETSI TS 103 301 V2.1.1 (2021). *Facilities layer protocols and communication requirements for infrastructure services* (SPATEM/MAPEM).
- ETSI TS 103 324 V2.1.1 (2023). *Collective Perception Service; Release 2*. https://www.etsi.org/deliver/etsi_ts/103300_103399/103324/02.01.01_60/ts_103324v020101p.pdf
- ETSI EN 302 663 V1.3.1 (2020) (ITS-G5 access layer) and ETSI EN 303 613 V1.1.1 (2020) (LTE-V2X access layer).
- ETSI TS 103 097 V2.1.1 (2021). *Security header and certificate formats*; ETSI TS 102 941 V2.2.1 (2022). *Trust and Privacy Management*.
- IEEE Std 1609.2. *Security Services for Applications and Management Messages* (WAVE).
- SAE J2735 (*V2X Communications Message Set Dictionary*), SAE J2945/1 (*On-Board System Requirements for V2V Safety Communications*), SAE J3224 (*V2X Sensor-Sharing for Cooperative and Automated Driving*).
- FCC (2024). *Use of the 5.850–5.925 GHz Band*, Second Report and Order, FCC 24-123. https://docs.fcc.gov/public/attachments/FCC-24-123A1_Rcd.pdf
- ISO/SAE 21434:2021. *Road vehicles — Cybersecurity engineering*.

---
**Navigation:** ← [ODD specification and runtime monitoring](01_odd_and_runtime_monitoring.ipynb) · [Module index](README.md) · [Industry deployment](03_industry_deployment.ipynb) →